In [3]:
import numpy as np
from itertools import combinations

def make_data(N, m, n, cov="toeplitz", rho=0.0, snr=5.0,
              beta_mode="uniform", rng=None, random_support=True):
    rng = np.random.default_rng(rng)

    # predictors
    idx = np.arange(m)
    if cov == "toeplitz":
        Sigma = rho ** np.abs(idx[:, None] - idx[None, :])
    elif cov == "equi":
        Sigma = (1 - rho) * np.eye(m) + rho * np.ones((m, m))
    else:
        raise ValueError(f"unknown cov: {cov!r}")
    L = np.linalg.cholesky(Sigma)
    X = rng.standard_normal((N, m)) @ L.T

    # support and coefficients
    S = np.sort(rng.choice(m, n, replace=False)) if random_support else np.arange(n)
    beta = np.zeros(m)
    if beta_mode == "uniform":
        beta[S] = rng.choice([-1, 1], n) * rng.uniform(0.5, 2.0, n)
    elif beta_mode == "ones":
        beta[S] = 1.0
    elif beta_mode == "decay":
        beta[S] = 1.0 / np.arange(1, n + 1)
    else:
        raise ValueError(f"unknown beta_mode: {beta_mode!r}")

    # noise scaled to hit target SNR
    signal_var = beta @ Sigma @ beta   # reduces to beta @ beta when Sigma = I
    sigma = np.sqrt(signal_var / snr)
    y = X @ beta + sigma * rng.standard_normal(N)
    return X, y, beta, S, sigma


def criteria(X, y, subset, sigma2_full):
    N = len(y)
    k = len(subset)
    if k == 0:
        rss = y @ y
    else:
        Xs = X[:, subset]
        coef, *_ = np.linalg.lstsq(Xs, y, rcond=None)
        r = y - Xs @ coef
        rss = r @ r
    return {
        "AIC": N * np.log(rss / N) + 2 * k,
        "BIC": N * np.log(rss / N) + k * np.log(N),
        "Cp":  rss / sigma2_full - N + 2 * k,
    }


# def select_best(X, y):
#     N, m = X.shape
#     # sigma^2 from the full model
#     coef, *_ = np.linalg.lstsq(X, y, rcond=None)
#     r = y - X @ coef
#     sigma2_full = r @ r / (N - m)

#     best = {}
#     scores = {"AIC": np.inf, "BIC": np.inf, "Cp": np.inf}
#     for k in range(m + 1):                      # all-subsets (feasible for m <~ 20)
#         for sub in combinations(range(m), k):
#             c = criteria(X, y, list(sub), sigma2_full)
#             for name in scores:
#                 if c[name] < scores[name]:
#                     scores[name] = c[name]
#                     best[name] = set(sub)
#     return best


def monte_carlo(R=500, **kw):
    hits = {"AIC": 0, "BIC": 0, "Cp": 0}
    sizes = {"AIC": [], "BIC": [], "Cp": []}
    for r in range(R):
        X, y, beta, S, _ = make_data(rng=r, **kw)
        best = select_best(X, y)
        for name, sel in best.items():
            hits[name] += (sel == set(S))
            sizes[name].append(len(sel))
    return {k: hits[k] / R for k in hits}, {k: np.mean(v) for k, v in sizes.items()}

# Example
# print(monte_carlo(R=200, N=100, m=8, n=3, snr=4.0))

In [13]:
X, y, beta, S, sigma = make_data(10, 5, 2, snr=4)

In [14]:
from modelSelector.model import LinearRegressionModel
from modelSelector.criterion import AIC, AIC_c, BIC, Mallows_Cp

# generate the full model, and fit it to data.
model = LinearRegressionModel('M_11111')
model.fit(data_x=X, data_y=y)

# criterion = Mallows_Cp(model)
# criterion = BIC()
criterion = AIC()

ModuleNotFoundError: No module named 'modelSelector'